In [ ]:
import os, cv2, shutil
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# 🔧 Paramètres
img_size = (128, 128)
epochs = 20
dataset_path = "/kaggle/input/skin-cancer-isic-images"
working_path = "/kaggle/working/cleaned_dataset"
FNFP_path = "/kaggle/working/FN_FP"
label_dict = {"benign": 0, "malignant": 1}

# 🧼 Préparer les dossiers
if os.path.exists(working_path): shutil.rmtree(working_path)
shutil.copytree(dataset_path, working_path)
os.makedirs(os.path.join(FNFP_path, "benign"), exist_ok=True)
os.makedirs(os.path.join(FNFP_path, "malignant"), exist_ok=True)

# 🧠 Calcule gradient
def compute_gradient(image):
    image = (image * 255).astype(np.uint8)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)
    grad_norm = cv2.normalize(grad_magnitude, None, 0, 1.0, cv2.NORM_MINMAX)
    return grad_norm

# 📥 Chargement des images avec gradient
def load_data_with_gradients(base_dir):
    X, X_grad, y, paths = [], [], [], []
    for label in ["benign", "malignant"]:
        folder = os.path.join(base_dir, label)
        for img_name in os.listdir(folder):
            path = os.path.join(folder, img_name)
            img = cv2.imread(path)
            if img is not None:
                img = cv2.resize(img, img_size) / 255.0
                grad = compute_gradient(img)
                X.append(img)
                X_grad.append(np.expand_dims(grad, axis=-1))
                y.append(label_dict[label])
                paths.append(path)
    return np.array(X), np.array(X_grad), to_categorical(y, 2), paths


# ✅ Nouveau modèle image + gradient
def build_gradient_cnn(input_shape):
    input_img = Input(shape=input_shape, name="input_image")
    input_grad = Input(shape=(input_shape[0], input_shape[1], 1), name="input_gradient")

    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
    x = layers.BatchNormalization()(x)

    g = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_grad)
    g = layers.BatchNormalization()(g)

    fusion = layers.Add()([x, g])
    fusion = layers.MaxPooling2D((2, 2))(fusion)

    fusion = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(fusion)
    fusion = layers.BatchNormalization()(fusion)
    fusion = layers.MaxPooling2D((2, 2))(fusion)

    fusion = layers.Flatten()(fusion)
    fusion = layers.Dense(128, activation='relu')(fusion)
    fusion = layers.BatchNormalization()(fusion)
    fusion = layers.Dropout(0.5)(fusion)

    output = layers.Dense(2, activation='softmax')(fusion)

    model = models.Model(inputs=[input_img, input_grad], outputs=output)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
# 📦 Charger les données
X, X_grad, y, paths = load_data_with_gradients(working_path)

# 🔀 Division une seule fois
X_train, X_test, X_train_grad, X_test_grad, y_train, y_test, train_paths, test_paths = train_test_split(
    X, X_grad, y, paths, test_size=0.2, stratify=np.argmax(y, axis=1), random_state=42
)
print(f"Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test: {X_test.shape}, Labels: {y_test.shape}")


In [ ]:
# 📊 Dictionnaire des erreurs
error_counter = defaultdict(int)

for run in range(3):
    print(f"\n🧠 Entraînement {run+1}/3")

    model = build_gradient_cnn((128, 128, 3))
    model.fit([X_train, X_train_grad], y_train,
              validation_data=([X_test, X_test_grad], y_test),
              epochs=epochs, batch_size=32,
              callbacks=[ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.2, verbose=0)],
              verbose=0)
    
    # ✅ Évaluation finale
    y_pred = np.argmax(model.predict([X_test, X_test_grad]), axis=1)
    y_true = np.argmax(y_test, axis=1)
    # Calcul des métriques
    test_accuracy = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred)
    test_recall = recall_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred)

    # Affichage des résultats
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test Precision: {test_precision * 100:.2f}%")
    print(f"Test Recall: {test_recall * 100:.2f}%")
    print(f"Test F1-Score: {test_f1 * 100:.2f}%")

    for i in range(len(y_true)):
        if y_pred[i] != y_true[i]:
            error_counter[test_paths[i]] += 1


In [ ]:
removed = 0

for path in list(error_counter.keys()):
    if error_counter[path] >= 3 and os.path.exists(path):
        
        # 🏷️ Déduire le label à partir du chemin
        if "benign" in path.lower():
            label = "benign"
        elif "malignant" in path.lower():
            label = "malignant"
        else:
            continue  # Si on ne peut pas identifier la classe, on ignore

        # 💾 Sauvegarder dans le dossier FN_FP approprié
        dest_path = os.path.join(FNFP_path, label, os.path.basename(path))
        shutil.copy(path, dest_path)

        # 🗑️ Supprimer de la base nettoyée
        os.remove(path)
        removed += 1

print(f"\n🧹 {removed} images mal classées 3 fois ont été supprimées de {working_path}.")
print(f"📁 Et sauvegardées dans : {FNFP_path}/benign et {FNFP_path}/malignant")


In [ ]:
# 📦 Charger les données
X, X_grad, y, paths = load_data_with_gradients(working_path)

# 🔀 Division une seule fois
X_train, X_test, X_train_grad, X_test_grad, y_train, y_test, train_paths, test_paths = train_test_split(
    X, X_grad, y, paths, test_size=0.2, stratify=np.argmax(y, axis=1), random_state=42
)
print(f"Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test: {X_test.shape}, Labels: {y_test.shape}")


In [ ]:
# 📊 Dictionnaire des erreurs
error_counter = defaultdict(int)

for run in range(3):
    print(f"\n🧠 Entraînement {run+1}/3")

    model = build_gradient_cnn((128, 128, 3))
    model.fit([X_train, X_train_grad], y_train,
              validation_data=([X_test, X_test_grad], y_test),
              epochs=epochs, batch_size=32,
              callbacks=[ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.2, verbose=0)],
              verbose=0)
    
    # ✅ Évaluation finale
    y_pred = np.argmax(model.predict([X_test, X_test_grad]), axis=1)
    y_true = np.argmax(y_test, axis=1)
    # Calcul des métriques
    test_accuracy = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred)
    test_recall = recall_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred)

    # Affichage des résultats
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test Precision: {test_precision * 100:.2f}%")
    print(f"Test Recall: {test_recall * 100:.2f}%")
    print(f"Test F1-Score: {test_f1 * 100:.2f}%")

    for i in range(len(y_true)):
        if y_pred[i] != y_true[i]:
            error_counter[test_paths[i]] += 1


In [ ]:
removed = 0

for path in list(error_counter.keys()):
    if error_counter[path] >= 3 and os.path.exists(path):
        
        # 🏷️ Déduire le label à partir du chemin
        if "benign" in path.lower():
            label = "benign"
        elif "malignant" in path.lower():
            label = "malignant"
        else:
            continue  # Si on ne peut pas identifier la classe, on ignore

        # 💾 Sauvegarder dans le dossier FN_FP approprié
        dest_path = os.path.join(FNFP_path, label, os.path.basename(path))
        shutil.copy(path, dest_path)

        # 🗑️ Supprimer de la base nettoyée
        os.remove(path)
        removed += 1

print(f"\n🧹 {removed} images mal classées 3 fois ont été supprimées de {working_path}.")
print(f"📁 Et sauvegardées dans : {FNFP_path}/benign et {FNFP_path}/malignant")


In [ ]:
# 📦 Charger les données
X, X_grad, y, paths = load_data_with_gradients(working_path)

# 🔀 Division une seule fois
X_train, X_test, X_train_grad, X_test_grad, y_train, y_test, train_paths, test_paths = train_test_split(
    X, X_grad, y, paths, test_size=0.2, stratify=np.argmax(y, axis=1), random_state=42
)
print(f"Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test: {X_test.shape}, Labels: {y_test.shape}")


In [ ]:
# 📊 Dictionnaire des erreurs
error_counter = defaultdict(int)

for run in range(3):
    print(f"\n🧠 Entraînement {run+1}/3")

    model = build_gradient_cnn((128, 128, 3))
    model.fit([X_train, X_train_grad], y_train,
              validation_data=([X_test, X_test_grad], y_test),
              epochs=epochs, batch_size=32,
              callbacks=[ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.2, verbose=0)],
              verbose=0)
    
    # ✅ Évaluation finale
    y_pred = np.argmax(model.predict([X_test, X_test_grad]), axis=1)
    y_true = np.argmax(y_test, axis=1)
    # Calcul des métriques
    test_accuracy = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred)
    test_recall = recall_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred)

    # Affichage des résultats
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test Precision: {test_precision * 100:.2f}%")
    print(f"Test Recall: {test_recall * 100:.2f}%")
    print(f"Test F1-Score: {test_f1 * 100:.2f}%")

    for i in range(len(y_true)):
        if y_pred[i] != y_true[i]:
            error_counter[test_paths[i]] += 1


In [ ]:
removed = 0

for path in list(error_counter.keys()):
    if error_counter[path] >= 3 and os.path.exists(path):
        
        # 🏷️ Déduire le label à partir du chemin
        if "benign" in path.lower():
            label = "benign"
        elif "malignant" in path.lower():
            label = "malignant"
        else:
            continue  # Si on ne peut pas identifier la classe, on ignore

        # 💾 Sauvegarder dans le dossier FN_FP approprié
        dest_path = os.path.join(FNFP_path, label, os.path.basename(path))
        shutil.copy(path, dest_path)

        # 🗑️ Supprimer de la base nettoyée
        os.remove(path)
        removed += 1

print(f"\n🧹 {removed} images mal classées 3 fois ont été supprimées de {working_path}.")
print(f"📁 Et sauvegardées dans : {FNFP_path}/benign et {FNFP_path}/malignant")


In [ ]:
# 📦 Charger les données
X, X_grad, y, paths = load_data_with_gradients(working_path)

# 🔀 Division une seule fois
X_train, X_test, X_train_grad, X_test_grad, y_train, y_test, train_paths, test_paths = train_test_split(
    X, X_grad, y, paths, test_size=0.2, stratify=np.argmax(y, axis=1), random_state=42
)
print(f"Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test: {X_test.shape}, Labels: {y_test.shape}")


In [ ]:
# 📊 Dictionnaire des erreurs
error_counter = defaultdict(int)

for run in range(3):
    print(f"\n🧠 Entraînement {run+1}/3")

    model = build_gradient_cnn((128, 128, 3))
    model.fit([X_train, X_train_grad], y_train,
              validation_data=([X_test, X_test_grad], y_test),
              epochs=epochs, batch_size=32,
              callbacks=[ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.2, verbose=0)],
              verbose=0)
    
    # ✅ Évaluation finale
    y_pred = np.argmax(model.predict([X_test, X_test_grad]), axis=1)
    y_true = np.argmax(y_test, axis=1)
    # Calcul des métriques
    test_accuracy = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred)
    test_recall = recall_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred)

    # Affichage des résultats
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test Precision: {test_precision * 100:.2f}%")
    print(f"Test Recall: {test_recall * 100:.2f}%")
    print(f"Test F1-Score: {test_f1 * 100:.2f}%")

    for i in range(len(y_true)):
        if y_pred[i] != y_true[i]:
            error_counter[test_paths[i]] += 1


In [ ]:
removed = 0

for path in list(error_counter.keys()):
    if error_counter[path] >= 3 and os.path.exists(path):
        
        # 🏷️ Déduire le label à partir du chemin
        if "benign" in path.lower():
            label = "benign"
        elif "malignant" in path.lower():
            label = "malignant"
        else:
            continue  # Si on ne peut pas identifier la classe, on ignore

        # 💾 Sauvegarder dans le dossier FN_FP approprié
        dest_path = os.path.join(FNFP_path, label, os.path.basename(path))
        shutil.copy(path, dest_path)

        # 🗑️ Supprimer de la base nettoyée
        os.remove(path)
        removed += 1

print(f"\n🧹 {removed} images mal classées 3 fois ont été supprimées de {working_path}.")
print(f"📁 Et sauvegardées dans : {FNFP_path}/benign et {FNFP_path}/malignant")


In [ ]:
# 📦 Charger les données
X, X_grad, y, paths = load_data_with_gradients(working_path)

# 🔀 Division une seule fois
X_train, X_test, X_train_grad, X_test_grad, y_train, y_test, train_paths, test_paths = train_test_split(
    X, X_grad, y, paths, test_size=0.2, stratify=np.argmax(y, axis=1), random_state=42
)
print(f"Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test: {X_test.shape}, Labels: {y_test.shape}")


In [ ]:
# 📊 Dictionnaire des erreurs
error_counter = defaultdict(int)

for run in range(3):
    print(f"\n🧠 Entraînement {run+1}/3")

    model = build_gradient_cnn((128, 128, 3))
    model.fit([X_train, X_train_grad], y_train,
              validation_data=([X_test, X_test_grad], y_test),
              epochs=epochs, batch_size=32,
              callbacks=[ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.2, verbose=0)],
              verbose=0)
    
    # ✅ Évaluation finale
    y_pred = np.argmax(model.predict([X_test, X_test_grad]), axis=1)
    y_true = np.argmax(y_test, axis=1)
    # Calcul des métriques
    test_accuracy = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred)
    test_recall = recall_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred)

    # Affichage des résultats
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test Precision: {test_precision * 100:.2f}%")
    print(f"Test Recall: {test_recall * 100:.2f}%")
    print(f"Test F1-Score: {test_f1 * 100:.2f}%")

    for i in range(len(y_true)):
        if y_pred[i] != y_true[i]:
            error_counter[test_paths[i]] += 1


In [ ]:
removed = 0

for path in list(error_counter.keys()):
    if error_counter[path] >= 3 and os.path.exists(path):
        
        # 🏷️ Déduire le label à partir du chemin
        if "benign" in path.lower():
            label = "benign"
        elif "malignant" in path.lower():
            label = "malignant"
        else:
            continue  # Si on ne peut pas identifier la classe, on ignore

        # 💾 Sauvegarder dans le dossier FN_FP approprié
        dest_path = os.path.join(FNFP_path, label, os.path.basename(path))
        shutil.copy(path, dest_path)

        # 🗑️ Supprimer de la base nettoyée
        os.remove(path)
        removed += 1

print(f"\n🧹 {removed} images mal classées 3 fois ont été supprimées de {working_path}.")
print(f"📁 Et sauvegardées dans : {FNFP_path}/benign et {FNFP_path}/malignant")


In [ ]:
# 📦 Charger les données
X, X_grad, y, paths = load_data_with_gradients(working_path)

# 🔀 Division une seule fois
X_train, X_test, X_train_grad, X_test_grad, y_train, y_test, train_paths, test_paths = train_test_split(
    X, X_grad, y, paths, test_size=0.2, stratify=np.argmax(y, axis=1), random_state=42
)
print(f"Train: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test: {X_test.shape}, Labels: {y_test.shape}")


In [ ]:
X_fnfp, X_fnfp_grad, y_fnfp, _ = load_data_with_gradients(FNFP_path)

print(f"\n📦 Nombre d'images récupérées depuis FN_FP : {X_fnfp.shape[0]}")

# 🔗 Fusion avec les données d'entraînement
X_train = np.concatenate([X_train, X_fnfp])
X_train_grad = np.concatenate([X_train_grad, X_fnfp_grad])
y_train = np.concatenate([y_train, y_fnfp])


print(f"\n🧠 Nouvelle taille de X_train : {X_train.shape}")
print(f"Train: {X_train.shape}, Labels: {y_train.shape}")

X_train = np.concatenate([X_train, X_fnfp])
X_train_grad = np.concatenate([X_train_grad, X_fnfp_grad])
y_train = np.concatenate([y_train, y_fnfp])


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

# 🔁 Réentraînement sur base enrichie
model = build_gradient_cnn((128, 128, 3))

# 📌 Callbacks
checkpoint = ModelCheckpoint(
    "best_model.keras", 
    monitor="val_accuracy", 
    save_best_only=True, 
    mode="max", 
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", 
    patience=3, 
    factor=0.2, 
    verbose=1
)

history = model.fit(
    [X_train, X_train_grad], y_train,
    validation_data=([X_test, X_test_grad], y_test),
    epochs=epochs,
    batch_size=32,
    callbacks=[checkpoint, reduce_lr],
    verbose=1
)

# 🔄 Charger le meilleur modèle sauvegardé
from tensorflow.keras.models import load_model
model = load_model("best_model.keras")

# ✅ Accuracies finales
train_accuracy = history.history['accuracy'][-1]
val_accuracy = history.history['val_accuracy'][-1]
print(f"\nTraining Accuracy: {train_accuracy * 100:.2f}%")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

# ✅ Évaluation finale
y_pred = np.argmax(model.predict([X_test, X_test_grad]), axis=1)
y_true = np.argmax(y_test, axis=1)

print("\n📋 Final Classification Report:")
print(classification_report(y_true, y_pred))

print(f"\n🎯 Test Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")
print(f"🎯 Test Precision: {precision_score(y_true, y_pred) * 100:.2f}%")
print(f"🎯 Test Recall: {recall_score(y_true, y_pred) * 100:.2f}%")
print(f"🎯 Test F1-Score: {f1_score(y_true, y_pred) * 100:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')

plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Sauvegarde de l'image
plt.savefig('accuracy_curve.png', dpi=300, bbox_inches='tight')

plt.show()
